In [1]:
import numpy as np
import pandas as pd
import os

In [2]:
def read_conllu(file_path):
    sentences = []
    labels = []
    
    sentence = []
    label = []
    
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("#") or line.strip() == "":
                if sentence:
                    sentences.append(sentence)
                    labels.append(label)
                    sentence = []
                    label = []
            else:
                parts = line.strip().split("\t")
                
                # Skip special tokens
                if "-" in parts[0] or "." in parts[0]:
                    continue
                
                word = parts[1]
                pos = parts[3]   # POS tag
                
                sentence.append(word)
                label.append(pos)
    
    return sentences, labels

In [3]:
train_sentences, train_labels = read_conllu("en_ewt-ud-train.conllu")

In [4]:
print(train_sentences[0])
print(train_labels[0])

['Al', '-', 'Zaman', ':', 'American', 'forces', 'killed', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.']
['PROPN', 'PUNCT', 'PROPN', 'PUNCT', 'ADJ', 'NOUN', 'VERB', 'PROPN', 'PROPN', 'PROPN', 'PUNCT', 'PROPN', 'PUNCT', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN', 'ADP', 'PROPN', 'PUNCT', 'ADP', 'DET', 'ADJ', 'NOUN', 'PUNCT']


In [5]:
unique_labels = list(set(label for sublist in train_labels for label in sublist))
print(unique_labels)
print("Total labels:", len(unique_labels))

['NOUN', 'DET', 'PRON', 'CCONJ', 'ADJ', 'ADV', 'ADP', 'PROPN', 'NUM', 'INTJ', 'SCONJ', 'PUNCT', 'X', 'SYM', 'PART', 'AUX', 'VERB']
Total labels: 17


In [6]:
!pip install transformers

  Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.8.0
    Uninstalling huggingface_hub-1.8.0:
      Successfully uninstalled huggingface_hub-1.8.0


In [12]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [13]:
label_list = list(set(label for sublist in train_labels for label in sublist))
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}
print(label2id)

{'NOUN': 0, 'DET': 1, 'PRON': 2, 'CCONJ': 3, 'ADJ': 4, 'ADV': 5, 'ADP': 6, 'PROPN': 7, 'NUM': 8, 'INTJ': 9, 'SCONJ': 10, 'PUNCT': 11, 'X': 12, 'SYM': 13, 'PART': 14, 'AUX': 15, 'VERB': 16}


In [14]:
def tokenize_and_align_labels(sentences, labels):
    tokenized_inputs = tokenizer(
        sentences,
        truncation=True,
        padding=True,
        is_split_into_words=True
    )
    
    aligned_labels = []
    
    for i, label in enumerate(labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None
        
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100)
            
            previous_word_idx = word_idx
        
        aligned_labels.append(label_ids)
    
    tokenized_inputs["labels"] = aligned_labels
    
    return tokenized_inputs

In [15]:
train_encodings = tokenize_and_align_labels(train_sentences, train_labels)

In [16]:
from transformers import AutoModelForTokenClassification
import torch

In [17]:
num_labels = len(label_list)
print(num_labels)

17


In [18]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

C:\Users\rutuj\anaconda3\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForTokenClassification: ['cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint 

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(device)

cpu


In [20]:
!pip install seqeval

In [21]:
from transformers import TrainingArguments, Trainer
from sklearn.model_selection import train_test_split

In [22]:
train_idx, val_idx = train_test_split(range(len(train_sentences)), test_size=0.1)

train_data = {key: [train_encodings[key][i] for i in train_idx] for key in train_encodings}
val_data = {key: [train_encodings[key][i] for i in val_idx] for key in train_encodings}

In [23]:
class POSDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings["input_ids"])

In [24]:
train_dataset = POSDataset(train_data)
val_dataset = POSDataset(val_data)

In [25]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="no"
)

In [31]:
train_dataset = torch.utils.data.Subset(train_dataset, range(300))
val_dataset = torch.utils.data.Subset(val_dataset, range(50))

In [32]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

In [33]:
trainer.train()

C:\Users\rutuj\anaconda3\lib\site-packages\transformers\optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Step,Training Loss
10,1.272000
20,1.061000
30,0.913000
40,0.701900
50,0.789600
60,0.638500
70,0.594700


TrainOutput(global_step=75, training_loss=0.8373526668548584, metrics={'train_runtime': 236.9766, 'train_samples_per_second': 1.266, 'train_steps_per_second': 0.316, 'total_flos': 26949882326400.0, 'train_loss': 0.8373526668548584, 'epoch': 1.0})

In [34]:
!pip install seqeval
from seqeval.metrics import classification_report, f1_score

In [35]:
predictions, labels, _ = trainer.predict(val_dataset)

In [36]:
preds = np.argmax(predictions, axis=2)

In [37]:
true_labels = []
true_preds = []

for i in range(len(labels)):
    temp_true = []
    temp_pred = []
    
    for j in range(len(labels[i])):
        if labels[i][j] != -100:
            temp_true.append(id2label[labels[i][j]])
            temp_pred.append(id2label[preds[i][j]])
    
    true_labels.append(temp_true)
    true_preds.append(temp_pred)

In [38]:
print("F1 Score:", f1_score(true_labels, true_preds))
print("\nClassification Report:\n")
print(classification_report(true_labels, true_preds))

F1 Score: 0.8911022576361223

Classification Report:

              precision    recall  f1-score   support

         ART       1.00      0.52      0.69        23
        CONJ       0.95      0.82      0.88        44
          DJ       0.76      0.80      0.78        40
          DP       0.96      0.97      0.96        69
          DV       0.61      0.73      0.67        30
         ERB       0.97      0.91      0.94        98
          ET       0.92      0.96      0.94        72
         NTJ       0.00      0.00      0.00         3
         OUN       0.89      0.92      0.90       131
         RON       0.96      0.97      0.96        90
        ROPN       0.69      0.81      0.75        27
          UM       1.00      0.40      0.57        10
        UNCT       0.97      0.98      0.98        63
          UX       0.84      0.96      0.90        51
          YM       0.00      0.00      0.00         6

   micro avg       0.90      0.89      0.89       757
   macro avg       0.77   

C:\Users\rutuj\anaconda3\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: PRON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\rutuj\anaconda3\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: AUX seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\rutuj\anaconda3\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\rutuj\anaconda3\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\rutuj\anaconda3\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\rutuj\anaconda3\lib\site-packages\seqeval\metrics\sequenc

In [39]:
sentence = ["John", "works", "at", "Google"]

In [40]:
inputs = tokenizer(sentence, return_tensors="pt", is_split_into_words=True)

In [41]:
with torch.no_grad():
    outputs = model(**inputs)
    
logits = outputs.logits
predictions = torch.argmax(logits, dim=2)

In [42]:
predicted_labels = [id2label[p.item()] for p in predictions[0]]
print(list(zip(sentence, predicted_labels)))

[('John', 'PUNCT'), ('works', 'PROPN'), ('at', 'VERB'), ('Google', 'ADP')]


## Differences between POS Tagging and Chunking

Part-of-Speech (POS) Tagging and Chunking are both important tasks in Natural Language Processing.

- POS Tagging assigns a grammatical category to each word in a sentence, such as noun, verb, adjective, etc.
  Example:
  "John works at Google"
  → John (NOUN), works (VERB), Google (PROPN)

- Chunking, also known as shallow parsing, groups words into meaningful phrases like noun phrases (NP) and verb phrases (VP).
  Example:
  "The quick brown fox"
  → [The quick brown fox] (Noun Phrase)

- POS Tagging focuses on individual words, while Chunking focuses on groups of words.
- POS Tagging is simpler, whereas Chunking captures higher-level sentence structure.

---

## Challenges Faced

- Handling tokenization and aligning labels with subword tokens was difficult.
- BERT splits words into multiple sub-tokens, which required careful handling using special labels (-100).
- Training time was high due to limited computational resources (CPU instead of GPU).
- Selecting and preparing the correct dataset format (.conllu) required understanding of data structure.
- Managing model performance with limited data samples was challenging.

---

## Observations and Insights

- Transformer models like BERT are powerful for sequence labeling tasks such as POS tagging.
- Proper preprocessing and label alignment are critical for model performance.
- Even with a small dataset, the model can learn basic patterns.
- Increasing dataset size and training epochs can significantly improve accuracy.
- Token classification tasks require careful handling of padding and ignored tokens.

Overall, this project helped in understanding how modern NLP models perform token-level classification tasks effectively.

(Note on Prediction Accuracy
The predicted labels are slightly incorrect due to limited training conditions. The model was trained on a small subset of data (300 samples) for only one epoch and on CPU, which restricts its learning capacity.
With more training data, additional epochs, and better computational resources, the model performance can be significantly improved.)